# FinSentinel — 金融交易实时欺诈检测平台
## 完整流程演示

本 Notebook 演示从数据生成到告警输出的完整端到端流程。

**前置条件**:
- Docker Compose 全栈已启动 (`docker-compose up -d`)
- Kafka Topic `txns` 已创建
- Python 依赖已安装 (`pip install -r requirements.txt`)

## Cell 1: 环境检查 — 验证连通性

In [ ]:
import sys, os
project_root = os.path.abspath('')
for _ in range(3):
    if os.path.exists(os.path.join(project_root, 'config', 'settings.py')):
        break
    project_root = os.path.dirname(project_root)
sys.path.insert(0, project_root)
os.chdir(project_root)

from config.settings import get_config
from kafka import KafkaAdminClient
from pymongo import MongoClient

config = get_config()

# 检查 Kafka
try:
    admin = KafkaAdminClient(bootstrap_servers=config['kafka']['bootstrap_servers'])
    topics = admin.list_topics()
    print(f'[OK] Kafka connected. Topics: {topics}')
    assert config['kafka']['topic'] in topics, f"Topic '{config['kafka']['topic']}' not found!"
    admin.close()
except Exception as e:
    print(f'[FAIL] Kafka: {e}')

# 检查 MongoDB
try:
    client = MongoClient(config['mongo']['uri'], serverSelectionTimeoutMS=5000)
    client.admin.command('ping')
    print(f'[OK] MongoDB connected: {config["mongo"]["uri"]}')
    client.close()
except Exception as e:
    print(f'[FAIL] MongoDB: {e}')

## Cell 2: 查看当前 Delta Lake 存储状态

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PipelineDemo")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

raw_path = config['delta']['raw_path']
clean_path = config['delta']['clean_path']

# 检查 Raw 表
try:
    raw_df = spark.read.format('delta').load(raw_path)
    print(f'[Raw Delta] {raw_df.count()} records')
    raw_df.show(3, truncate=False)
except Exception as e:
    print(f'[Raw Delta] Table not found or empty: {e}')

# 检查 Clean 表
try:
    clean_df = spark.read.format('delta').load(clean_path)
    print(f'\n[Clean Delta] {clean_df.count()} records')
    clean_df.select('user_hash', 'ip_mask', 'amount').show(3, truncate=False)
except Exception as e:
    print(f'[Clean Delta] Table not found or empty: {e}')

## Cell 3: 查询 MongoDB 当前告警

In [ ]:
from pymongo import MongoClient

client = MongoClient(config['mongo']['uri'])
db = client[config['mongo']['database']]
collection = db[config['mongo']['collection']]

alerts = list(collection.find({}).sort('alert_time', -1).limit(5))
print(f'Total alerts in DB: {collection.count_documents({})}')
print(f'Latest {len(alerts)} alerts:')
for a in alerts:
    print(f"  [{a.get('rule_type')}] {a.get('transaction_id')} | ${a.get('amount')} | {a.get('alert_time')}")

client.close()

## Cell 4: 启动数据生成器（正常模式）

> 在终端/后台运行，或在 Notebook 中作为子进程启动

In [ ]:
import subprocess, time, signal

# 启动生成器子进程
generator_proc = subprocess.Popen(
    ['python', 'src/transaction_generator.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(f'Generator PID: {generator_proc.pid}')
print('Generator running... (will be stopped automatically)')

# 让它运行几秒钟产生数据
time.sleep(5)

## Cell 5: 注入一笔 R3 深夜大额欺诈交易

> 触发条件: amount > $5000 且时间在 00:00-05:00

In [ ]:
fraud_proc = subprocess.run(
    ['python', 'src/transaction_generator.py', '--inject-fraud', 'R3'],
    capture_output=True,
    text=True
)
print(fraud_proc.stdout)
if fraud_proc.stderr:
    print(f'STDERR: {fraud_proc.stderr}')

## Cell 6: 注入一笔 R4 商户高频欺诈交易

> 触发条件: 同一商户 1 分钟内交易 > 3 笔

In [ ]:
# 连续注入多笔同一商户交易
for i in range(4):
    result = subprocess.run(
     ['python', 'src/transaction_generator.py', '--inject-fraud', 'R4'],
        capture_output=True,
        text=True
    )
    print(f"R4 injection {i+1}: {result.stdout.strip()}")
    time.sleep(0.5)

## Cell 7: 验证告警 — MongoDB 查询

In [ ]:
import json
from pymongo import MongoClient

client = MongoClient(config['mongo']['uri'])
collection = client[config['mongo']['database']][config['mongo']['collection']]

time.sleep(3)  # 等待流处理

alerts = list(collection.find({}).sort('alert_time', -1).limit(10))
print(f'=== 最新告警 ({len(alerts)} 条) ===\n')

for alert in alerts:
    print(f"[ALERT] {alert['rule_type']}")
    print(f"  Transaction: {alert['transaction_id']}")
    print(f"  Amount: ${alert['amount']:.2f}")
    print(f"  Alert Time: {alert['alert_time']}")
    if 'trigger_detail' in alert:
        print(f"  Detail: {json.dumps(alert['trigger_detail'], default=str)}")
    print()

client.close()

## Cell 8: 验证 Delta Lake 存储

In [ ]:
# Raw Delta Table — 验证原始数据已落地
raw_df = spark.read.format('delta').load(raw_path)
raw_count = raw_df.count()
print(f'Raw Delta Table: {raw_count} records')
print('Sample (last 3):')
raw_df.select('transaction_id', 'user_id', 'amount', 'ingest_time').sort('ingest_time', ascending=False).show(3, truncate=False)

# Clean Delta Table — 验证 PII 已脱敏
clean_df = spark.read.format('delta').load(clean_path)
clean_count = clean_df.count()
print(f'\nClean Delta Table: {clean_count} records')
print('Sample (last 3, PII masked):')
clean_df.select('transaction_id', 'user_hash', 'ip_mask', 'amount', 'ingest_time').sort('ingest_time', ascending=False).show(3, truncate=False)

# 验证脱敏效果
print('\n=== PII 脱敏验证 ===')
raw_user_id = raw_df.select('user_id').first()
clean_user_hash = clean_df.select('user_hash').first()
if raw_user_id and clean_user_hash:
    print(f'Raw user_id example: {raw_user_id[0]}')
    print(f'Clean user_hash example: {clean_user_hash[0]}')
    print(f'user_id is hashed: {raw_user_id[0] not in clean_user_hash[0]}')
    print(f'device_id removed from Clean: {"device_id" not in clean_df.columns}')

## Cell 9: 清理 — 停止生成器、关闭 Spark

In [ ]:
# 停止生成器
if generator_proc.poll() is None:
    generator_proc.send_signal(signal.SIGTERM)
    generator_proc.wait(timeout=5)
    print('Generator stopped.')

# 关闭 Spark
spark.stop()
print('Spark session closed.')
print('\n=== Pipeline Demo Complete ===')